### Task: Given 26th March 2021 demand - clean dataset based on business team rules; we need to forecast demand for first 3 (30mins internal) demand value of 27th March 2021.

In [14]:
import pandas as pd
import numpy as np
from joblib import load, dump
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from math import sqrt, ceil, floor
from datetime import datetime, timedelta

In [15]:
def round_timestamp_30interval(x):
    if type(x)==str:
        x = datetime.strptime(x, '%Y-%m-%d %H:%M:%S')
    return x- timedelta(minutes=x.minute%30, seconds=x.second, microseconds=x.microsecond)

def time_features(data):
    data['mins'] = data.ts.dt.minute
    data['hour'] = data.ts.dt.hour
    data['day'] = data.ts.dt.day
    data['month'] = data.ts.dt.month
    data['dayofweek'] = data.ts.dt.dayofweek
    data['quarter'] = data.ts.dt.quarter
    return data

def prediction_without_lag(df):
    return predict_without_lag.predict(df[['pickup_cluster','mins','hour','month','quarter','dayofweek']])

def prediction_with_lag(df):
    return predict_with_lag.predict(df[['pickup_cluster', 'mins', 'hour', 'month', 'quarter',
           'dayofweek', 'lag_1', 'lag_2', 'lag_3','rolling_mean']])

def shift_with_lag_and_rollingmean(df):
    df = df.sort_values(by=['pickup_cluster', 'ts']).drop_duplicates(subset=['ts', 'pickup_cluster'])
    df = df.set_index(['ts', 'pickup_cluster', 'mins', 'hour', 'month', 'quarter', 'dayofweek'])
    df['lag_1'] = df.groupby(level=['pickup_cluster'])['request_count'].shift(1)
    df['lag_2'] = df.groupby(level=['pickup_cluster'])['request_count'].shift(2)
    df['lag_3'] = df.groupby(level=['pickup_cluster'])['request_count'].shift(3)
    df['rolling_mean'] = (
        df.groupby(level=['pickup_cluster'])['request_count']
        .transform(lambda x: x.rolling(window=3).mean().shift(1))
    )
    df = df.reset_index(drop=False).dropna()
    df = df[['ts', 'pickup_cluster', 'mins', 'hour', 'month', 'quarter',
             'dayofweek', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean', 'request_count']]
    return df


In [16]:
df = pd.read_csv('C:/Sachin/project/Bike-Taxi-Rides-Request-Demand-Forecast/data/test_dataset/cleaned_test_booking_data.csv', compression = 'gzip', low_memory=False)
cluster_model = load('C:/Sachin/project/Bike-Taxi-Rides-Request-Demand-Forecast/output/pickup_cluster_model.joblib')
predict_without_lag = load('C:/Sachin/project/Bike-Taxi-Rides-Request-Demand-Forecast/output/prediction_model_without_lag.joblib')
predict_with_lag = load('C:/Sachin/project/Bike-Taxi-Rides-Request-Demand-Forecast/output/prediction_model.joblib')


### Use Clustering Kmeans Model for Geospacial Feature - `pickup_cluster`

In [17]:
df['pickup_cluster'] = cluster_model.predict(df[['pick_lat','pick_lng']])
df.head(10)

C:\Users\Binish\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but MiniBatchKMeans was fitted without feature names
  warnings.warn(


,ts,number,pick_lat,pick_lng,drop_lat,drop_lng,pickup_cluster
0,2021-03-26 06:49:38,-1,12.903468,77.637080,12.916259,77.675476,31
1,2021-03-26 15:14:23,0,12.903838,77.591774,12.890039,77.593720,1
2,2021-03-26 15:57:32,6,12.963516,77.674740,12.912828,77.627310,8
3,2021-03-26 23:34:53,7,12.944017,77.564270,12.967625,77.608060,12
4,2021-03-26 23:45:56,9,12.983270,77.752070,12.963221,77.748400,44
5,2021-03-26 18:54:05,11,12.919469,77.671100,12.933288,77.607310,48
6,2021-03-26 18:42:49,15,12.947335,77.684310,12.974627,77.606064,46
7,2021-03-26 23:14:56,15,12.979332,77.640590,12.947475,77.684230,43
8,2021-03-26 10:59:13,17,12.923716,77.607410,12.922842,77.593240,34
9,2021-03-26 16:44:09,53,12.888448,77.577240,12.937987,77.568726,24


### Data preparation and processing

In [18]:

df['ts'] = np.vectorize(round_timestamp_30interval)(df['ts'])
df['ts'] = pd.to_datetime(df['ts'])

df = df[['ts','number','pickup_cluster']]
df = df.groupby(by = ['ts','pickup_cluster']).count().reset_index()
df.columns = ['ts','pickup_cluster','request_count']

## Adding Dummy pickup cluster -1

## Change this Data based on your data
l = [datetime(2021,3,26,00,00,00) + timedelta(minutes = 30*i) for i in range(0,51)]
lt = []
for x in l:
    lt.append([x, -1, 0])
temp = pd.DataFrame(lt, columns = ['ts','pickup_cluster','request_count'])
df = pd.concat([df, temp], ignore_index=True)

data = df.set_index(['ts', 'pickup_cluster']).unstack().fillna(value=0).asfreq(freq='30Min').stack().sort_index(level=1).reset_index()

# Removing Dummy Cluster
data = data[data.pickup_cluster>=0]

df = time_features(data)

In [19]:
# Model without Lag (past data) requirement
data_without_lag = df[df['ts'] >= datetime(2021, 3, 27, 0, 0, 0)].copy()
data_without_lag['request_count'] = prediction_without_lag(data_without_lag)
print(data_without_lag)

                      ts  pickup_cluster  request_count  mins  hour  day  \
99   2021-03-27 00:00:00               0       0.247233     0     0   27   
100  2021-03-27 00:30:00               0       0.145774    30     0   27   
101  2021-03-27 01:00:00               0       0.122551     0     1   27   
150  2021-03-27 00:00:00               1       0.268162     0     0   27   
151  2021-03-27 00:30:00               1       0.180283    30     0   27   
...                  ...             ...            ...   ...   ...  ...   
2548 2021-03-27 00:30:00              48       0.186719    30     0   27   
2549 2021-03-27 01:00:00              48       0.108549     0     1   27   
2598 2021-03-27 00:00:00              49       0.166667     0     0   27   
2599 2021-03-27 00:30:00              49       0.131050    30     0   27   
2600 2021-03-27 01:00:00              49       0.064775     0     1   27   

      month  dayofweek  quarter  
99        3          5        1  
100       3        

In [20]:
data_without_lag.to_csv('C:/Sachin/project/Bike-Taxi-Rides-Request-Demand-Forecast/Data/prediction_without_lag_model.csv',index = False, compression = 'gzip')

### Using Iteration 3 - Best Model with Lag Features and Rolling Means (Recursive Multi-Step Forecast used)

In [21]:
# Model with Lag - Recursive Multi-Step Forecast
start_date = datetime(2021, 3, 27, 0, 0, 0)
for x in range(3):
    df = shift_with_lag_and_rollingmean(df)
    df.loc[df[df['ts'] == start_date + timedelta(minutes=30 * x)].index, 'request_count'] = \
        prediction_with_lag(df[df['ts'] == start_date + timedelta(minutes=30 * x)])

In [22]:
data_with_lag = df[df['ts']>=datetime(2021,3,27,00,00,00)].__copy__()
data_with_lag

,ts,pickup_cluster,mins,hour,month,quarter,dayofweek,lag_1,lag_2,lag_3,rolling_mean,request_count
42,2021-03-27 00:00:00,0,0,0,3,1,5,1.000000,6.000000,10.0,5.666667,1.425921
43,2021-03-27 00:30:00,0,30,0,3,1,5,1.425921,1.000000,6.0,2.808640,0.016801
44,2021-03-27 01:00:00,0,0,1,3,1,5,0.016801,1.425921,1.0,0.814241,0.152825
87,2021-03-27 00:00:00,1,0,0,3,1,5,3.000000,10.000000,17.0,10.000000,0.754189
88,2021-03-27 00:30:00,1,30,0,3,1,5,0.754189,3.000000,10.0,4.584730,1.731260
...,...,...,...,...,...,...,...,...,...,...,...,...
2203,2021-03-27 00:30:00,48,30,0,3,1,5,1.816677,7.000000,10.0,6.272226,2.152014
2204,2021-03-27 01:00:00,48,0,1,3,1,5,2.152014,1.816677,7.0,3.656230,1.702260
2247,2021-03-27 00:00:00,49,0,0,3,1,5,1.000000,1.000000,0.0,0.666667,0.179792
2248,2021-03-27 00:30:00,49,30,0,3,1,5,0.179792,1.000000,1.0,0.726597,0.027412


In [23]:
data_with_lag.to_csv('C:/Sachin/project/Bike-Taxi-Rides-Request-Demand-Forecast/Data/prediction_with_lag_model.csv',index = False, compression = 'gzip')